In [1]:
import os

os.environ[
    "TF_USE_LEGACY_KERAS"
] = "1"

In [3]:
!pip install -q tensorflow-recommenders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 3.5 MB/s eta 0:00:00


In [4]:
import pandas as pd
import pickle

import tensorflow as tf
import tensorflow_recommenders as tfrs

In [6]:
training_df = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/trainds/training_dataset.parquet"
)

with open(
    "/kaggle/input/datasets/selinparmar/trainds/feature_vocab.pkl",
    "rb"
) as file:

    feature_vocab = pickle.load(file)

In [7]:
training_df[
    'article_id'
] = (
    training_df[
        'article_id'
    ].astype(str)
)

In [8]:
item_vocab = (
    feature_vocab[
        'item_vocab'
    ]
)

product_type_vocab = (
    feature_vocab[
        'product_type_vocab'
    ]
)

season_vocab = (
    feature_vocab[
        'season_vocab'
    ]
)

In [9]:
class CandidateTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Article embedding
        self.article_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=item_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(item_vocab) + 1,
                    32
                )
            ])
        )

        # Product type embedding
        self.product_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    product_type_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        product_type_vocab
                    ) + 1,
                    16
                )
            ])
        )

        # Season embedding
        self.season_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    season_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        season_vocab
                    ) + 1,
                    8
                )
            ])
        )

        # Dense network
        self.dense_layers = (
            tf.keras.Sequential([

                tf.keras.layers.Dense(
                    128,
                    activation='relu'
                ),

                tf.keras.layers.Dense(
                    64
                )
            ])
        )

    def call(
        self,
        features
    ):

        article_vector = (
            self.article_embedding(
                features[
                    'article_id'
                ]
            )
        )

        product_vector = (
            self.product_embedding(
                features[
                    'product_type_name'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features[
                    'season'
                ]
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features[
                    'purchase_count'
                ],
                tf.float32
            )

        ], axis=1)

        combined_features = (
            tf.concat([

                article_vector,
                product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

        return (
            self.dense_layers(
                combined_features
            )
        )

In [10]:
candidate_model = (
    CandidateTower()
)

print(
    "Candidate Tower built!"
)

2026-05-18 05:46:41.576228: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Candidate Tower built!


In [11]:
sample_item_features = {

    'article_id':
        tf.constant(
            training_df[
                'article_id'
            ].head(5)
        ),

    'product_type_name':
        tf.constant(
            training_df[
                'product_type_name'
            ].head(5)
        ),

    'season':
        tf.constant(
            training_df[
                'season'
            ].head(5)
        ),

    'purchase_count':
        tf.constant(
            training_df[
                'purchase_count'
            ].head(5),
            dtype=tf.float32
        )
}

In [12]:
candidate_embeddings = (
    candidate_model(
        sample_item_features
    )
)

print(
    candidate_embeddings.shape
)

(5, 64)
